## 1. Import Python Libraries

In [12]:
# import crim_intervals and specific modules
import crim_intervals
from crim_intervals import * 
from crim_intervals import main_objs
import crim_intervals.visualizations as viz
import crim_intervals.corpus_tools as corpus_tools
import music_utils
from music_utils import *
# import other libraries used in the notebooks
import pandas as pd
from IPython.display import display
from pyvis.network import Network
import altair as alt
import plotly_express as px
import os
import re
import seaborn as sns
from collections import Counter

# for use in Jupyter notebook, create a local folder for music files
MYDIR = ("saved_csv")
CHECK_FOLDER = os.path.isdir(MYDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MYDIR)
    print("created folder : ", MYDIR)
else:
    print(MYDIR, "folder already exists.")
    
MUSDIR = ("Music_Files")
CHECK_FOLDER = os.path.isdir(MUSDIR)

# If folder doesn't exist, then create it.
if not CHECK_FOLDER:
    os.makedirs(MUSDIR)
    print("created folder : ", MUSDIR)

else:
    print(MUSDIR, "folder already exists.")

# code for Quarto and the charting tools
alt.renderers.enable('default')
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook"

import warnings
warnings.filterwarnings('ignore')


    

saved_csv folder already exists.
Music_Files folder already exists.


In [13]:
# function for ngram heatmap:  do not edit!

def ngram_heatmap(piece, combine_unisons_choice, kind_choice, directed, compound, length_choice, include_count, entries_only):
    # find entries for model
    nr = piece.notes(combineUnisons = combine_unisons_choice)
    mel = piece.melodic(df = nr, 
                        kind = kind_choice,
                        directed = directed,
                        compound = compound,
                        end = False)
    
    # this is for entries only
    if entries_only == True:    
        # pass the following ngrams to the plot below as first df
        entry_ngrams = piece.entries(df = mel, 
                                    n = length_choice, 
                                    thematic = True, 
                                    anywhere = True,
                                    exclude = ['Rest'])
        # pass the ngram durations below to the plot as second df
        entry_ngrams_duration = piece.durations(df = mel, 
                                            n =length_choice, 
                                            mask_df = entry_ngrams)
        # make the heatmap
        chart = viz.plot_ngrams_heatmap(entry_ngrams, 
                                         entry_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[],  
                                         includeCount=include_count)
        
        return chart
    
    # this is for ALL mel ngrams (if entries is False in form)
    else:
        mel_ngrams = piece.ngrams(df = mel, n = length_choice, exclude = ['Rest'])
        mel_ngrams_duration = piece.durations(df = mel, 
                                          n =length_choice)
        
        chart = viz.plot_ngrams_heatmap(mel_ngrams, 
                                         mel_ngrams_duration, 
                                         selected_patterns=[], 
                                         voices=[], 
                                         includeCount=include_count)
        
        # # mel_ngrams_detail = piece.detailIndex(mel_ngrams, offset = False)  
        return chart

#define the function to convert tuples to strings
def convert_tuple(tup):
    out = ""
    if isinstance(tup, tuple):
        out = ', '.join(tup)
    return out  
# Create a color map dictionary that maps each cadence type to a specific color
cadence_color_dict = {
    'Authentic': '#1f77b4',  # blue
    'Clausula Vera': '#ff7f0e',  # orange
    'Evaded Authentic': '#2ca02c',  # green
    'Leaping Contratenor': '#d62728',  # red
    'Phrygian Clausula Vera': '#9467bd',  # purple
    'Phrygian': '#8c564b',  # brown
    'Evaded Clausula Vera': '#e377c2',  # pink
    'Double Leading Tone': '#7f7f7f',  # gray
    'Abandoned Clausula Vera': '#bcbd22',  # olive
    'Quince': '#17becf',  # cyan
    'Altizans Only': '#aec7e8',  # light blue
    'Abandoned Authentic': '#ffbb78'  # light orange
}

# List of values for CVFs
values = ['Cx', 'CB', 'CTB', 'TCB', 'zCB', 'CT', 'CTA', 'CTx', 'TC', 'TCA',
          'TCx', 'ACT', 'CAT', 'CTb', 'TCb', 'CTL', 'CzB', 'CxT', 'zCT',
          'Ct', 'tCx']

# Get the default color palette
color_palette = px.colors.qualitative.Plotly

# Create a color dictionary
cvf_color_dict = {value: color for value, color in zip(values, color_palette)}


# tone order for charts

custom_tone_order = ['E-', 'B-', 'F', 'C', 'G', 'D', 'A', 'E', 'B']  # Your desired o


def radar_note_plot(weighted_notes_df, pitch_class_order='Fifths',
                    exclude_rests=True, limit_to_active=True, color_grouping='title'):
    if exclude_rests:
        weighted_notes_df = weighted_notes_df[weighted_notes_df['pitch_class'] != 'Rest']
    if limit_to_active:
        weighted_notes_df = weighted_notes_df[weighted_notes_df['scaled'] > 0]

    pc_order = NOTE_PC_ORDERS[pitch_class_order]
    active_pcs = [pc for pc in pc_order if pc in weighted_notes_df['pitch_class'].values]

    weighted_notes_df = weighted_notes_df.copy()
    weighted_notes_df['pitch_class'] = pd.Categorical(
        weighted_notes_df['pitch_class'], categories=active_pcs, ordered=True
    )
    weighted_notes_df = weighted_notes_df.sort_values('pitch_class')

    fig = px.line_polar(
        weighted_notes_df,
        r='scaled',
        theta='pitch_class',
        color=color_grouping,
        line_close=True,
        range_r=[0, weighted_notes_df['scaled'].max() * 1.1],
        markers=True,
        category_orders={
            'pitch_class': active_pcs,
            color_grouping: list(weighted_notes_df[color_grouping].unique())
        },
        color_discrete_sequence=contrasting_colors[:len(weighted_notes_df[color_grouping].unique())]
    )
    fig.update_traces(fill='toself', mode='markers+lines', opacity=.7)
    fig.update_layout(
        showlegend=True,
        legend=dict(title=color_grouping, orientation='h', yanchor='bottom', y=-0.5, xanchor='right', x=1),
        height=600, width=600,
        title=f'Weighted Note Distribution in Corpus ({pitch_class_order} order)'
    )
    fig.show()
    fig.write_html("charts/radar_note_plot.html")


def cadence_radar(cadences, tone_ordering='Thirds', limit_to_active=False):
    FIFTHS    = {'C':0,'G':1,'D':2,'A':3,'E':4,'B':5,'F#':6,'C#':7,'A-':8,'E-':9,'B-':10,'F':11}
    THIRDS    = {'C':0,'E':1,'G':2,'B-':3,'D':4,'F':5,'A':6,'C#':7}
    CHROMATIC = {'C':0,'C#':1,'D-':1,'D':2,'E-':3,'E':4,'F':5,'F#':6,'G-':6,'G':7,'A-':8,'A':9,'B-':10,'B':11}
    CATEGORY_ORDER = FIFTHS if tone_ordering == 'Fifths' else THIRDS if tone_ordering == 'Thirds' else CHROMATIC

    grouped = cadences.groupby(['Title', 'Tone']).size().reset_index(name='count')
    title_sums = grouped.groupby('Title')['count'].sum()
    grouped['Percentage'] = (grouped['count'] / grouped['Title'].map(title_sums)) * 100

    tone_order  = sorted(CATEGORY_ORDER.keys(), key=lambda x: CATEGORY_ORDER[x])
    active_tones = [t for t in tone_order if t in grouped['Tone'].values] if limit_to_active else tone_order

    full_index = pd.MultiIndex.from_product([grouped['Title'].unique(), active_tones], names=['Title', 'Tone'])
    grouped = (
        grouped[grouped['count'] > 0]
        .set_index(['Title', 'Tone'])[['count', 'Percentage']]
        .reindex(full_index, fill_value=0)
        .reset_index()
    )
    if limit_to_active:
        grouped = grouped[grouped['count'] > 0]

    tone_rank = {t: i for i, t in enumerate(active_tones)}
    grouped = grouped.assign(_sort=grouped['Tone'].map(tone_rank)).sort_values(['Title', '_sort']).drop(columns='_sort').reset_index(drop=True)
    grouped['Tone'] = pd.Categorical(grouped['Tone'], categories=active_tones, ordered=True)

    fig = px.line_polar(grouped, r='Percentage', theta='Tone', line_close=True,
                        color='Title', markers=True, category_orders={'Tone': active_tones})
    fig.update_traces(fill='toself', line=dict(width=2))
    fig.update_layout(
        width=800, height=600,
        legend=dict(orientation='h', yanchor='bottom', y=-0.4, xanchor='center', x=0.5,
                    title=dict(text='Titles', side='top', font_size=12),
                    itemsizing='constant', itemwidth=30,
                    bordercolor='black', borderwidth=1, bgcolor='rgba(255,255,255,0.8)'),
        title=dict(text=f'Relative Distribution of Cadence Tones in Corpus ordered by {tone_ordering}', x=0.5),
        polar=dict(radialaxis=dict(visible=True, title='Percentage')),
        margin=dict(b=120)
    )
    fig.show()
    fig.write_html("charts/radar_cadence_plot.html")


# =============================================================================
# VISUALIZATION: SPECIES + CADENCES
# =============================================================================

def plot_species_cadences(species_df, cadences_df, composer, title):
    """
    Plot species intervals and cadences by progress in a two-panel chart.

    Args:
        species_df:   output of find_species()
        cadences_df:  piece.cadences() with 'Composer' and 'Title' columns added
        composer:     piece composer string (for chart title)
        title:        piece title string (for chart title)
    """
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=('Species: Octave by Progress', 'Cadences: Tone by Progress')
    )

    # Normalize Progress to 0–1 for both datasets so they share the same x-axis scale
    species_df  = species_df.copy()
    cadences_df = cadences_df.copy()
    for df, col in [(species_df, 'Progress'), (cadences_df, 'Progress')]:
        lo, hi = df[col].min(), df[col].max()
        if hi > lo:
            df[col] = (df[col] - lo) / (hi - lo)

    octave_order = ['EbBbEb', 'BbFBb', 'FCF', 'CGC', 'GDG', 'DAD', 'AEA', 'EBE', 'BF#B']
    fig_sp = px.scatter(
        species_df, x='Progress', y='Octave', color='Voice',
        hover_data=['composer', 'title', 'Measure', 'Interval', 'Species'],
        color_discrete_sequence=contrasting_colors[:len(species_df['Voice'].unique())],
        category_orders={'Octave': octave_order}
    )
    for trace in fig_sp.data:
        trace.showlegend = True
        fig.add_trace(trace, row=1, col=1)
    fig.update_yaxes(title_text='Octave', categoryorder='array', categoryarray=octave_order, row=1, col=1)

    tone_order = ['B-', 'F', 'C', 'G', 'D', 'A', 'E']
    fig_cad = px.scatter(
        cadences_df, x='Progress', y='Tone', color='CadType',
        hover_data=['Composer', 'Title', 'CadType', 'Tone'],
        color_discrete_sequence=contrasting_colors[len(species_df['Voice'].unique()):],
        category_orders={'Tone': tone_order}
    )
    for trace in fig_cad.data:
        trace.marker.symbol = 'diamond'
        trace.marker.size = 10
        trace.marker.opacity = 0.8
        trace.showlegend = True
        fig.add_trace(trace, row=2, col=1)

    fig.update_traces(marker=dict(size=8, opacity=0.7), row=1, col=1)
    fig.update_yaxes(title_text='Tone', categoryorder='array', categoryarray=tone_order, row=2, col=1)
    fig.update_yaxes(title_text='Octave', tickmode='linear', dtick=1, row=1, col=1)

    x_min = min(species_df['Progress'].min(), cadences_df['Progress'].min())
    x_max = max(species_df['Progress'].max(), cadences_df['Progress'].max())
    fig.update_xaxes(range=[x_min, x_max], autorange=False)
    fig.update_xaxes(title_text='Progress', row=2, col=1)
    fig.update_layout(
        height=600, width=800,
        legend=dict(x=1.02, y=1.0, xanchor='left', yanchor='top'),
        margin=dict(r=200),
        title=f'Species and Cadences by Progress in {composer} : {title}',
        legend_title='Legend'
    )
    fig.show()


# Voice Range Chart


def find_mode_range(df, final_value, voice_value, top_n=7):
    """
    Finds the range of notes in a given voice to help distinguish modal types.

    df:          corpus notes DataFrame (from corpus_note_durs with pitch_class=False)
    final_value: the final tone of each piece
    voice_value: which voice to check
    top_n:       number of highest-percentage notes to use for range calculation
    """
    filtered_df = df[
        (df['Final'] == final_value) &
        (df['Voice'] == voice_value) &
        (df['Notes'] != 'Rest')
    ]

    top_durations = filtered_df.groupby(['Title', 'Voice'], observed=True).apply(
    lambda x: x.nlargest(top_n, 'Percentage'), include_groups=False).reset_index(level=['Title', 'Voice']).reset_index(drop=True)

    # Build note-position mapping
    base_notes = ['C', 'D', 'E', 'F', 'G', 'A', 'B', 'B-']
    octaves = list(range(2, 6))
    note_positions = {}
    position = 0
    for octave in octaves:
        for note in base_notes:
            note_positions[f"{note}{octave}"]  = position
            note_positions[f"{note}-{octave}"] = position - 0.5
            note_positions[f"{note}b{octave}"] = position - 0.5
            note_positions[f"{note}#{octave}"] = position + 0.5
            position += 1

    def get_note_position(note):
        if note in note_positions:
            return note_positions[note]
        if '-' in note:
            parts = note.split('-')
            if len(parts) == 2:
                alt = f"{parts[0]}-{parts[1]}"
                if alt in note_positions:
                    return note_positions[alt]
        if '#' in note:
            parts = note.split('#')
            if len(parts) == 2:
                alt = f"{parts[0]}#{parts[1]}"
                if alt in note_positions:
                    return note_positions[alt]
        print(f"Warning: Could not determine position for note: {note}")
        return -1000

    top_durations['NotePosition'] = top_durations['Notes'].apply(get_note_position)

    result = []
    for (title, voice), group in top_durations.groupby(['Title', 'Voice'], observed=True):
        if len(group) > 0:
            sorted_group = group.sort_values('NotePosition')
            lowest_note  = sorted_group.iloc[0]['Notes']
            highest_note = sorted_group.iloc[-1]['Notes']
            title_rows   = df[df['Title'] == title]
            composer = title_rows['Composer'].iloc[0] if not title_rows.empty else 'Unknown'
            final    = title_rows['Final'].iloc[0]    if not title_rows.empty else 'Unknown'
            result.append({
                'Composer':    composer,
                'Title':       title,
                'Voice':       voice,
                'Final':       final,
                'LowestNote':  lowest_note,
                'HighestNote': highest_note,
                'Range':       f"{lowest_note} to {highest_note}"
            })

    return pd.DataFrame(result)


def voice_range_chart(corpus_pitch_data, recta_order, top_n=7):

    final_value = corpus_pitch_data['Final'].iloc[0]
    designated_voice = '2'

    # Run find_mode_range for every voice and combine
    voices = corpus_pitch_data['Voice'].unique()
    all_voice_ranges = []
    for voice in voices:
        voice_df = find_mode_range(corpus_pitch_data, final_value=final_value, voice_value=voice, top_n=top_n)
        if not voice_df.empty:
            all_voice_ranges.append(voice_df)

    corpus_voice_ranges = pd.concat(all_voice_ranges, ignore_index=True)

    # Add positional columns needed by the chart
    corpus_voice_ranges['Lowest_pos'] = corpus_voice_ranges['LowestNote'].apply(
        lambda n: recta_order.index(n) if n in recta_order else 0
    )
    corpus_voice_ranges['Highest_pos'] = corpus_voice_ranges['HighestNote'].apply(
        lambda n: recta_order.index(n) if n in recta_order else 0
    )
    corpus_voice_ranges['Range_width'] = (
        corpus_voice_ranges['Highest_pos'] - corpus_voice_ranges['Lowest_pos']
    )

    # Force voice to string for chart labels
    corpus_voice_ranges['Voice'] = corpus_voice_ranges['Voice'].astype(str)

    # Movement order: titles without a movement keyword are chanson models → rank 0
    mass_movements = ['Kyrie', 'Gloria', 'Credo', 'Sanctus', 'Agnus Dei']

    def title_rank(title):
        for i, movement in enumerate(mass_movements):
            if movement in title:
                return i + 1
        return 0  # no movement keyword → chanson model, sorts first

    # Find lowest note of designated voice per piece, for sorting within groups
    designated_voice_notes = {}
    for _, row in corpus_voice_ranges.iterrows():
        if row['Voice'] == designated_voice:
            designated_voice_notes[row['Title']] = row['LowestNote']

    sorted_titles = sorted(
        designated_voice_notes.keys(),
        key=lambda x: (
            title_rank(x),                                # primary: model first, then movements
            recta_order.index(designated_voice_notes[x])  # secondary: lowest note
        )
    )

    # Create combined Title+Voice label, voices sorted numerically within each piece
    corpus_voice_ranges['Title_Voice'] = (
        corpus_voice_ranges['Title'] + '  [V' + corpus_voice_ranges['Voice'] + ']'
    )

    sorted_title_voice = []
    for title in sorted_titles:
        piece_rows = corpus_voice_ranges[corpus_voice_ranges['Title'] == title]
        for voice in sorted(piece_rows['Voice'].unique(), key=lambda v: int(v)):
            label = f"{title}  [V{voice}]"
            if label not in sorted_title_voice:
                sorted_title_voice.append(label)

    # Create figure
    fig = px.bar(
        corpus_voice_ranges,
        x='Range_width',
        y='Title_Voice',
        base='Lowest_pos',
        orientation='h',
        color='Voice',
        color_discrete_map={
            '1': 'rgba(255, 99, 71, 0.5)',
            '2': 'rgba(54, 162, 235, 0.5)',
            '3': 'rgba(100, 20, 100, 0.3)'
        },
        hover_data=['Title', 'Voice', 'Range'],
        category_orders={'Title_Voice': sorted_title_voice},
        labels={'Title_Voice': 'Piece / Voice'}
    )

    _ref_by_final = {
        'D': ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'],
        'E': ['E3', 'C3', 'E4', 'C4', 'E5', 'C5'],
        'F': ['C3', 'F3', 'C4', 'F4', 'C5', 'F5'],
        'G': ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'],
        'A': ['E3', 'A3', 'E4', 'A4', 'E5', 'A5'],
        'C': ['C3', 'G3', 'C4', 'G4', 'C5', 'G5'],
    }
    reference_points = _ref_by_final.get(final_value, ['D3', 'G3', 'D4', 'G4', 'D5', 'G5'])
    shapes = []
    annotations = []
    for point in reference_points:
        if point in recta_order:
            x_position = recta_order.index(point)
            shapes.append(dict(
                type='line',
                x0=x_position, y0=0,
                x1=x_position, y1=len(sorted_title_voice),
                line=dict(dash='dot', color='gray', width=1)
            ))
            annotations.append(dict(
                x=x_position,
                y=len(sorted_title_voice) - 1,
                text=point,
                showarrow=False,
                font=dict(size=12)
            ))

    fig.update_layout(
        title=f'Voice Ranges in Pieces with Final {final_value}',
        xaxis=dict(
            title='Note Range',
            tickmode='array',
            tickvals=list(range(len(recta_order))),
            ticktext=recta_order,
            tickangle=45
        ),
        yaxis=dict(
            title='',
            autorange='reversed',
            categoryorder='array',
            categoryarray=sorted_title_voice
        ),
        height=900,
        width=1000,
        bargap=0.3,
        bargroupgap=0.1,
        margin=dict(l=250, r=100, t=100, b=50),
        shapes=shapes,
        annotations=annotations
    )

    fig.show()
    fig.write_html("charts/corpus_voice_ranges.html")

# =============================================================================
# CONSTANTS
# =============================================================================

pitch_order = ['A#1', 'B1',
    'C2', 'C#2', 'D2', 'D#2','E-2', 'E2', 'E#2', 'F2', 'F#2', 'G-2', 'G2', 'G#2', 'A-2', 'A2', 'A#2','B-2', 'B2', 'B#2',
    'C3', 'C#3', 'D-3','D3', 'D#3', 'E-3','E3', 'E#3', 'F3', 'F#3', 'G-3', 'F##3', 'G3', 'G#3', 'A-3', 'A3', 'A#3', 'B-3','B3', 'B#3',
    'C4', 'C#4', 'D-4','D4', 'D#4','E-4', 'E4', 'F-4', 'E#4', 'F4', 'F#4', 'G-4', 'F##4', 'G4', 'G#4', 'A-4','A4', 'A#4', 'B-4', 'B4', 'B#4',
    'C5', 'C#5','C##5', 'D-5','D5', 'D#5', 'E-5','E5', 'F-5','E#5','F5', 'F#5', 'G-5', 'F##5','G5', 'G#5', 'A-5', 'A5', 'A#5', 'B-5', 'B5',
    'C6']

recta_order = ['D2', 'E-2', 'E2', 'F2', 'F#2', 'G2', 'G#2', 'A2', 'B-2', 'B2',
               'C3', 'C#3','D3', 'D#3', 'E-3','E3', 'F3', 'F#3',  'G3', 'G#3', 'A-3', 'A3', 'B-3','B3',
               'C4', 'C#4', 'D-4','D4', 'D#4','E-4', 'E4', 'F4', 'F#4', 'G-4',  'G4', 'G#4', 'A-4','A4',  'B-4', 'B4',
               'C5', 'C#5','D-5','D5', 'D#5', 'E-5','E5','F5', 'F#5', 'G-5', 'G5', 'G#5', 'A-5', 'A5',  'B-5', 'B5',
               'C6']

pitch_class_order_no_rests = ['C', 'C#', 'D-','D', 'D#', 'E-', 'E', 'F-', 'E#', 'F', 'F#', 'G-', 'F##', 'G', 'G#', 'A-','A', 'A#', 'B-', 'B', 'B#']
pitch_class_order_with_rests = pitch_class_order_no_rests + ['Rest']
pitch_class_order = pitch_class_order_with_rests

category_order = {
    'C': 0, 'C#': 1, 'Db': 2, 'D': 3, 'D#': 4, 'Eb': 5, 'E': 6, 'F': 7, 'F#': 8,
    'G': 9, 'G#': 10, 'A': 11, 'A#': 12, 'Bb': 13, 'B': 14, 'Rest': 15
}

REST_TOKENS = {'r', 'rest', 'Rest', '-', ''}

NOTE_PC_CHROMATIC = ['C', 'C#', 'D-', 'D', 'D#', 'E-', 'E', 'F', 'F#', 'G-', 'G', 'G#', 'A-', 'A', 'A#', 'B-', 'B', 'F-', 'E#', 'F##', 'B#', 'Rest']
NOTE_PC_FIFTHS    = ['C', 'G', 'D', 'A', 'E', 'B', 'F#', 'C#', 'A-', 'E-', 'B-', 'F', 'G#', 'D#', 'A#', 'D-', 'G-', 'F-', 'E#', 'B#', 'F##', 'Rest']
NOTE_PC_DIATONIC  = ['C', 'D', 'E', 'F', 'G', 'A', 'B', 'B-', 'E-', 'A-', 'D-', 'C#', 'F#', 'G#', 'D#', 'A#', 'G-', 'F-', 'E#', 'F##', 'B#', 'Rest']
NOTE_PC_ORDERS    = {'Chromatic': NOTE_PC_CHROMATIC, 'Fifths': NOTE_PC_FIFTHS, 'Diatonic': NOTE_PC_DIATONIC}

contrasting_colors = [
    '#636EFA', '#DC267F', '#009E73', '#FFB000', '#977277',
    '#EC4899', '#48BB78', '#ED8936', '#2563EB', '#8338EC',
    '#FF922B', '#06D6A0', '#EF4444', '#F97316', '#84CC16',
    '#3B82F6', '#A855F7', '#22C55E', '#EA580C', '#94A3B8',
]

def standardize_note(note):
    if '-' in note:
        return note.replace('-', 'b')
    return note



## Import Score

In [14]:
# import a score

path = 'Music_Files/Ala_11_Exultate_unit2.xml'
piece = importScore(path)
 

piece.metadata

{'title': 'Music_Files/Ala_11_Exultate_unit2.xml',
 'composer': 'Giovanni Battista Ala',
 'date': None}

## 2. get notes, durs, measures, beats, and progress

In [15]:
nr = piece.notes().copy()
dur = piece.durations().copy()

# join them up, col by col

notes_durs = nr.astype(str) + "_" + dur.astype(str)
notes_durs.columns = [f'{col}_note-dur' for col in notes_durs.columns]

notes_durs = notes_durs.replace("nan_nan", np.nan)

# get the measure, beat, and progress information
nr_dur_det = piece.detailIndex(notes_durs, progress=True, offset=True)

# just a quick look!
nr_dur_det.head()


Alto_note-dur Tenore  1618/1633_note-dur  \
Measure Beat Offset Progress                                            
1.0     1.0  0.0    0.000000     Rest_24.0                     D4_3.0   
        2.5  3.0    0.008523           NaN                     G3_1.0   
        3.0  4.0    0.011364           NaN                     G3_1.0   
        3.5  5.0    0.014205           NaN                     G3_1.0   
        4.0  6.0    0.017045           NaN                    B-3_2.0   

                             Basso continuo  1633_note-dur  \
Measure Beat Offset Progress                                 
1.0     1.0  0.0    0.000000                        G2_8.0   
        2.5  3.0    0.008523                           NaN   
        3.0  4.0    0.011364                           NaN   
        3.5  5.0    0.014205                           NaN   
        4.0  6.0    0.017045                           NaN   

                             MusicXML Part_note-dur Part-5_note-dur  \
Measure Beat Offset Progress                                          
1.0     1.0  0.0    0.000000              Rest_24.0          D4_3.0   
        2.5  3.0    0.008523                    NaN          G3_1.0   
        3.0  4.0    0.011364                    NaN          G3_1.0   
        3.5  5.0    0.014205                    NaN          G3_1.0   
        4.0  6.0    0.017045                    NaN         B-3_2.0   

                             Partitura  1618_note-dur  
Measure Beat Offset Progress                           
1.0     1.0  0.0    0.000000                   G2_8.0  
        2.5  3.0    0.008523                      NaN  
        3.0  4.0    0.011364                      NaN  
        3.5  5.0    0.014205                      NaN  
        4.0  6.0    0.017045                      NaN

## Find the Bassus Parts

In [ ]:
## Check the Columns for the "B" parts 

# list of cols
col_list = nr_dur_det.columns

# check them for "B", that is saved as bassus_idx
for col in col_list:
    if col.startswith('B'):
        # find just the 'index' position of that column (which is an integer
        bassus_idx = col_list.get_loc(col)

# check column index
bassus_idx

2

In [50]:
# filter original df for just the "B" column and the LAST one, which is -1
nr_dur_bassus_parts = nr_dur_det.iloc[:, [bassus_idx, -1]].copy()

# look 'across' the df, and put all the values at each offset into a new column as a LIST
nr_dur_bassus_parts['combined'] = nr_dur_bassus_parts.values.tolist()
nr_dur_bassus_parts

Basso continuo  1633_note-dur  \
Measure Beat Offset Progress                                 
1.0     1.00 0.0    0.000000                        G2_8.0   
        2.50 3.0    0.008523                           NaN   
        3.00 4.0    0.011364                           NaN   
        3.50 5.0    0.014205                           NaN   
        4.00 6.0    0.017045                           NaN   
...                                                    ...   
44.0    3.00 348.0  0.988636                        D3_4.0   
        3.50 349.0  0.991477                           NaN   
        3.75 349.5  0.992898                           NaN   
        4.00 350.0  0.994318                           NaN   
45.0    1.00 352.0  1.000000                        G2_8.0   

                             Partitura  1618_note-dur          combined  
Measure Beat Offset Progress                                             
1.0     1.00 0.0    0.000000                   G2_8.0  [G2_8.0, G2_8.0]  
        2.50 3.0    0.008523                      NaN        [nan, nan]  
        3.00 4.0    0.011364                      NaN        [nan, nan]  
        3.50 5.0    0.014205                      NaN        [nan, nan]  
        4.00 6.0    0.017045                      NaN        [nan, nan]  
...                                               ...               ...  
44.0    3.00 348.0  0.988636                   D3_4.0  [D3_4.0, D3_4.0]  
        3.50 349.0  0.991477                      NaN        [nan, nan]  
        3.75 349.5  0.992898                      NaN        [nan, nan]  
        4.00 350.0  0.994318                      NaN        [nan, nan]  
45.0    1.00 352.0  1.000000                   G2_8.0  [G2_8.0, G2_8.0]  

[411 rows x 3 columns]

In [18]:
# get the LENGTH of the SET of values

nr_dur_bassus_parts['count_bassus_values'] = nr_dur_bassus_parts['combined'].apply(lambda x: len(set(x)))

In [19]:
# check that length.  If 1, then they MATCH and thus TRUE
nr_dur_bassus_parts['matching'] = nr_dur_bassus_parts['count_bassus_values'] == 1

# filtered df for the FALSE matches
bassus_diff_df = nr_dur_bassus_parts[nr_dur_bassus_parts['matching'] == False].copy()

# reset index so that the measure, beat, progress are cols
bassus_diff_df.reset_index(inplace=True)

# a quick look
bassus_diff_df

,Measure,Beat,Offset,Progress,Basso continuo 1633_note-dur,Partitura 1618_note-dur,combined,count_bassus_values,matching
0,5.0,1.0,32.0,0.090909,D3_4.0,D3_2.0,"[D3_4.0, D3_2.0]",2,False
1,5.0,2.0,34.0,0.096591,NaN,D3_2.0,"[nan, D3_2.0]",2,False
2,17.0,3.0,132.0,0.375000,B-2_4.0,B-2_3.0,"[B-2_4.0, B-2_3.0]",2,False
3,17.0,4.5,135.0,0.383523,NaN,B-2_1.0,"[nan, B-2_1.0]",2,False


In [21]:
# get the names of the columns, in this case from the original note_dur df

sel_cols = nr_dur_det.columns[[bassus_idx, -1]].to_list()

# use these to name v1 and v2 for the agg below, for clarity
v1 = sel_cols[0]
v2 = sel_cols[1]


# this function joins up the two elements in each list of values for the given voices
def seq(s):
    return ' '.join(s.dropna())

# group by on Measure, so we have ONE row per measure, with each voice's note_dur sequence
# first it sorts by offsets (so the values for each measure are in order
# then it groups things by measures--so they are distinct DFs
# then finally it 'aggregates', taking the value for each voice and appling the joining function
sigs = (bassus_diff_df.sort_values('Offset')
          .groupby('Measure')
          .agg(bc=(v1, seq), partitura=(v2, seq)))


# assemble the two variants into a single 'signature', which goes in a new column
sigs['signature'] = sigs['bc'] + ' | ' + sigs['partitura']

sigs

,bc,partitura,signature
Measure,,,
5.0,D3_4.0,D3_2.0 D3_2.0,D3_4.0 | D3_2.0 D3_2.0
17.0,B-2_4.0,B-2_3.0 B-2_1.0,B-2_4.0 | B-2_3.0 B-2_1.0


In [22]:
# vocabulary: each distinct variant, how often it occurs, and where
vocab = (sigs.reset_index()
             .groupby('signature')
             .agg(count=('Measure', 'size'), measures=('Measure', list))
             .sort_values('count', ascending=False)).reset_index()
vocab

,signature,count,measures
0,B-2_4.0 | B-2_3.0 B-2_1.0,1,[17.0]
1,D3_4.0 | D3_2.0 D3_2.0,1,[5.0]


## Render the Scores for these passages

In [23]:
# here we look 'down' the Measure column and save all the measure Numbers as a list

measure_list = bassus_diff_df['Measure'].tolist()

# and for each measure, we get one BEFORE and one AFTER
for measure in measure_list:
    start = int(measure) - 1
    stop = int(measure) + 1
    # now we print them
    piece.verovioPrintExample(start, stop)



Score:
File Name:  Ala_11_Exultate_unit2
Giovanni Battista Ala
Music_Files/Ala_11_Exultate_unit2.xml
Measures: 4-6


Score:
File Name:  Ala_11_Exultate_unit2
Giovanni Battista Ala
Music_Files/Ala_11_Exultate_unit2.xml
Measures: 4-6


Score:
File Name:  Ala_11_Exultate_unit2
Giovanni Battista Ala
Music_Files/Ala_11_Exultate_unit2.xml
Measures: 16-18


Score:
File Name:  Ala_11_Exultate_unit2
Giovanni Battista Ala
Music_Files/Ala_11_Exultate_unit2.xml
Measures: 16-18


## Checking Other Voice Pairs

#### Steps

- get voice pairs
- get durs
- check where the durs are different (this will highlight the passages where the partitura and vocal parts diverge
- get the offsets, or perhaps the range of these, then assemble the 'differences' between the full melodies in these places

In [ ]:
durs = piece.durations()
col_dict = dict.fromkeys(durs.columns)


voice_matching_dict = {'Alto': 'MusicXML Part',
 'Tenore  1618/1633': 'Part-5',
 'Basso continuo  1633': 'Partitura  1618'}

vocal_parts = voice_matching_dict.keys()

list_filtered_melted_dfs = []

for voice in vocal_parts:
    matching_voice = voice_matching_dict[voice]
    pair_durs_df = durs[[voice, matching_voice]]
    # print(voice, matching_voice)
    pair_durs_df.dropna(how='all', inplace=True)
    pair_durs_df.fillna("_", inplace=True)

    # dur_df = pair_durs_df

    voice_cols = [c for c in pair_durs_df.columns if c not in ('combined', 'count_dur_vals')]


    pair_durs_df['count_dur_vals'] = pair_durs_df[voice_cols].nunique(axis=1)

    filtered_pair_durs = pair_durs_df[pair_durs_df['count_dur_vals'] == 2]

    filtered_pair_durs_det = piece.detailIndex(filtered_pair_durs, offset=True, progress=True).reset_index()

    id_cols = ['Measure', 'Beat', 'Offset', 'Progress', 'count_dur_vals']
    value_cols = [col for col in filtered_pair_durs_det.columns if col not in id_cols]

    filtered_melted = filtered_pair_durs_det.melt(id_vars=id_cols,
                                        value_vars=value_cols,
                                        var_name='Voice',
                                        value_name='Duration')
    filtered_melted
    list_filtered_melted_dfs.append(filtered_melted)


results = pd.concat(list_filtered_melted_dfs)

results





,Measure,Beat,Offset,Progress,count_dur_vals,Voice,Duration
0,5.0,3.000,36.00,0.102273,2.0,Alto,0.75
1,5.0,3.375,36.75,0.104403,2.0,Alto,0.25
2,5.0,3.500,37.00,0.105114,2.0,Alto,0.5
3,5.0,3.750,37.50,0.106534,2.0,Alto,0.25
4,5.0,3.875,37.75,0.107244,2.0,Alto,0.25
...,...,...,...,...,...,...,...
3,17.0,4.500,135.00,0.383523,2.0,Basso continuo 1633,_
4,5.0,1.000,32.00,0.090909,2.0,Partitura 1618,2.0
5,5.0,2.000,34.00,0.096591,2.0,Partitura 1618,2.0
6,17.0,3.000,132.00,0.375000,2.0,Partitura 1618,3.0


In [84]:
voice_order = ['Alto', 'MusicXML Part',
 'Tenore  1618/1633', 'Part-5',
 'Basso continuo  1633', 'Partitura  1618']

fig = px.scatter(results,
                 x='Offset',
                 y='Voice',
                 category_orders={'Voice': voice_order})
fig.show()



In [47]:
# get the voice pairs
durs = piece.durations()
col_dict = dict.fromkeys(durs.columns)
col_dict

{'Alto': None,
 'Tenore  1618/1633': None,
 'Basso continuo  1633': None,
 'MusicXML Part': None,
 'Part-5': None,
 'Partitura  1618': None}

In [59]:
voice_matching_dict = {'Alto': 'MusicXML Part',
 'Tenore  1618/1633': 'Part-5',
 'Basso continuo  1633': 'Partitura  1618'}

vocal_parts = voice_matching_dict.keys()

for voice in vocal_parts:
    matching_voice = voice_matching_dict[voice]
    pair_durs_df = durs[[voice, matching_voice]]
    # print(voice, matching_voice)
pair_durs_df

,Basso continuo 1633,Partitura 1618
0.0,8.0,8.0
3.0,NaN,NaN
4.0,NaN,NaN
5.0,NaN,NaN
6.0,NaN,NaN
...,...,...
348.0,4.0,4.0
349.0,NaN,NaN
349.5,NaN,NaN
350.0,NaN,NaN
